# AlphaTrain Iteration 1c: Rebalanced Training (RunPod)

Train ResNet on **399K states**: 50% expert heuristic + 50% elite self-play (score>1000, T=0.1 sharpened).

**Key fixes from failed 1a/1b:**
- **Loss rebalancing**: val_weight=0.002 (value loss was 600x policy, destroying backbone features)
- **Elite filtering**: only self-play games scoring >1000 (213/493 games, removes blunders)
- **Train from scratch**: random init, no warm start (avoids distribution shift sensitivity)
- **20 epochs**

**Upload to `/workspace/`:**
1. `colorlines_selfplay_train.tar.gz` — code (81 KB)
2. `mixed_iter1c.pt.gz` — training data (129 MB compressed)

**Download results:** `best.pt` and `latest.pt` from `/workspace/checkpoints/selfplay_iter1c/`

In [ ]:
import os
os.chdir('/workspace')
!tar xzf colorlines_selfplay_train.tar.gz

os.makedirs('alphatrain/data', exist_ok=True)

# Decompress training data (129 MB gz -> 12.8 GB pt)
data_dst = 'alphatrain/data/mixed_iter1c.pt'
if not os.path.exists(data_dst):
    print('Decompressing mixed_iter1c.pt.gz...')
    !gunzip -c /workspace/mixed_iter1c.pt.gz > {data_dst}
print(f'Data: {os.path.getsize(data_dst)/1e9:.1f} GB')

!pip install -q numpy numba scipy pytest

In [ ]:
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

!python -m pytest alphatrain/tests/test_model.py alphatrain/tests/test_observation.py -v --tb=short 2>&1 | tail -10

In [ ]:
# Iteration 1c: rebalanced training from scratch
# Key fix: val_weight=0.002 (was 1.0, value loss was 600x policy)
# Elite self-play (score>1000) + expert heuristic, sharpened T=0.1
# Train from scratch (no --resume, no --warm-start)
!python -m alphatrain.train \
    --tensor-file alphatrain/data/mixed_iter1c.pt \
    --gpu-data --amp --compile \
    --value-bins 1 --val-weight 0.002 \
    --epochs 20 --batch-size 8192 --lr 1e-3 --warmup-epochs 3 \
    --save-dir /workspace/checkpoints/selfplay_iter1c

In [ ]:
# Evaluate: standalone policy score
# Baselines: epoch 6 heuristic=314, iter 1b mixed=245
!python -m alphatrain.evaluate --player policy \
    --model /workspace/checkpoints/selfplay_iter1c/best.pt \
    --games 50 --seed 42

In [ ]:
!ls -lh /workspace/checkpoints/selfplay_iter1c/